# 03 — YOLOv8 Training with augmentation
Sanity check with YOLOv8n (5 epochs), then full training with YOLOv8m (50 epochs).

In [3]:
from pathlib import Path

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "dataset").is_dir() and (_root / "src").is_dir():
        DATASET_DIR = _root / "dataset"
        break
else:
    raise FileNotFoundError("Repo root not found (need dataset/ and src/).")

# Prefer your preprocessed dataset if it exists; otherwise use dataset/
PREPROCESSED_DIR = DATASET_DIR / "bdd100k_preprocessing"
if PREPROCESSED_DIR.is_dir():
    DATA_DIR = PREPROCESSED_DIR
    print(f"Using preprocessed dataset: {DATA_DIR}")
else:
    DATA_DIR = DATASET_DIR
    print(f"Using dataset root: {DATA_DIR}")

Using dataset root: C:\Users\micha\Downloads\Object-Detection-main\dataset


In [4]:
!pip install ultralytics --no-deps

In [5]:
import sys
import os
from pathlib import Path
import numpy as np

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "src").is_dir() and (_root / "dataset").is_dir():
        break
else:
    raise FileNotFoundError("Repo root not found (need src/ and dataset/).")

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.utils import seed_everything, log_environment

seed_everything()
log_environment()

PyTorch: 2.11.0+cu128
Ultralytics: 8.4.42
GPU: NVIDIA GeForce RTX 5090
CUDA: 12.8


In [6]:
import os
import shutil
import re

DATA_CONFIG = _root / "configs" / "yolov8_bdd100k.yaml"
PROJECT_DIR = _root / "outputs" / "bdd100k_project" / "runs"

os.makedirs(PROJECT_DIR, exist_ok=True)

In [5]:
AUGMENTATION = dict(
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    translate=0.1,
    scale=0.5,
    degrees=0.0,
)

## Sanity Check — YOLOv8n (5 epochs)

In [6]:
from ultralytics import YOLO

model_n = YOLO("yolov8n.pt")

results_n = model_n.train(
    data=DATA_CONFIG,
    epochs=5,
    imgsz=640,
    batch=16,
    name="yolov8n_bdd100k_sanity",
    project=PROJECT_DIR,
    device=0,
    seed=42,
    **AUGMENTATION,
)

Ultralytics 8.4.42  Python-3.11.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\micha\Downloads\Object-Detection-main\configs\yolov8_bdd100k.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_bdd100k_sanity, nbs=64, nms=False, opset

In [7]:
print("Sanity check complete.")
print(f"Results saved to: {os.path.join(PROJECT_DIR, 'yolov8n_bdd100k_sanity')}")

import glob
sanity_dir = os.path.join(PROJECT_DIR, "yolov8n_bdd100k_sanity")
for f in sorted(glob.glob(os.path.join(sanity_dir, "*.png"))):
    print(f"  {os.path.basename(f)}")

Sanity check complete.
Results saved to: C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_project\runs\yolov8n_bdd100k_sanity
  BoxF1_curve.png
  BoxPR_curve.png
  BoxP_curve.png
  BoxR_curve.png
  confusion_matrix.png
  confusion_matrix_normalized.png
  results.png


## Full Training — YOLOv8m (50 epochs)

In [8]:
from ultralytics import YOLO
model_m = YOLO("yolov8m.pt")

results_m = model_m.train(
    data=DATA_CONFIG,
    epochs=50,
    imgsz=640,
    batch=16,
    name="yolov8m_bdd100k_v1",
    project=PROJECT_DIR,
    device=0,
    seed=42,
    patience=15,
    save=True,
    plots=True,
    **AUGMENTATION,
)

Ultralytics 8.4.42  Python-3.11.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\micha\Downloads\Object-Detection-main\configs\yolov8_bdd100k.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_bdd100k_v1, nbs=64, nms=False, opset=No

## Save Outputs

In [7]:
best_weights_src = os.path.join(PROJECT_DIR, "yolov8m_bdd100k_v1_augmentation/weights/best.pt")
results_csv_src = os.path.join(PROJECT_DIR, "yolov8m_bdd100k_v1_augmentation/results.csv")

trained_dir = os.path.join(PROJECT_DIR, "trained")
os.makedirs(trained_dir, exist_ok=True)

best_weights_dst = os.path.join(trained_dir, "yolov8m_bdd100k_best_augmentation.pt")
results_csv_dst = os.path.join(trained_dir, "yolov8m_results_augmentation.csv")

if os.path.exists(best_weights_src):
    shutil.copy(best_weights_src, best_weights_dst)
    print(f"Best weights saved to {best_weights_dst}")

if os.path.exists(results_csv_src):
    shutil.copy(results_csv_src, results_csv_dst)
    print(f"Results CSV saved to {results_csv_dst}")

run_dir = os.path.join(PROJECT_DIR, "yolov8m_bdd100k_v1")
if os.path.exists(run_dir):
    print(f"\nFull run directory: {run_dir}")
    for f in sorted(os.listdir(run_dir)):
        print(f"  {f}")

Best weights saved to C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_project\runs\trained\yolov8m_bdd100k_best_augmentation.pt
Results CSV saved to C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_project\runs\trained\yolov8m_results_augmentation.csv


In [10]:
import pandas as pd

results_csv = os.path.join(trained_dir, "yolov8m_results_augmentation.csv")
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f"Training completed: {len(df)} epochs")
    print(f"Best mAP50: {df['metrics/mAP50(B)'].max():.4f}")
    print(f"Best mAP50-95: {df['metrics/mAP50-95(B)'].max():.4f}")
    display(df.tail())

Training completed: 50 epochs
Best mAP50: 0.5474
Best mAP50-95: 0.3033


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
45,46,2806.61,1.20602,0.59604,0.93284,0.61724,0.51893,0.53482,0.29969,1.29794,0.70921,0.99271,0.000091,0.000091,0.000091
46,47,2866.25,1.20089,0.59203,0.93153,0.61571,0.51151,0.53533,0.29880,1.29822,0.70761,0.99107,0.000074,0.000074,0.000074
47,48,2926.01,1.20055,0.58778,0.93112,0.63748,0.50004,0.53414,0.29993,1.29544,0.70795,0.98969,0.000058,0.000058,0.000058
48,49,2980.71,1.19069,0.58096,0.92970,0.61350,0.50720,0.53574,0.29899,1.29582,0.70851,0.99088,0.000041,0.000041,0.000041
49,50,3034.36,1.18781,0.57900,0.92686,0.60559,0.52295,0.53736,0.29994,1.29504,0.70689,0.99138,0.000025,0.000025,0.000025


In [9]:
import shutil
from IPython.display import FileLink

folder_path = _root / "outputs" / "bdd100k_project"

zip_path = folder_path.with_suffix(".zip")

shutil.make_archive(str(folder_path), "zip", str(folder_path))

# Display a download link
FileLink(zip_path)

C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_project.zip